# Análisis de entrenamiento distribuido vía sockets

**Workers:** 2  
**Épocas:** 400  

Este cuaderno fue generado automáticamente al finalizar el entrenamiento.


## Arquitectura del sistema

- Servidor central que coordina el entrenamiento.
- Workers distribuidos conectados por sockets.
- El servidor envía parámetros globales.
- Cada worker calcula gradientes sobre su shard local.
- El servidor promedia gradientes y actualiza el modelo.


## Arquitectura del modelo

Red neuronal multicapa:

- Entrada: 784
- Capa oculta 1: 256 + LeakyReLU
- Capa oculta 2: 128 + LeakyReLU
- Salida: 10 + Softmax

Arquitectura usada: `[784, 256, 128, 10]`


## Carga de resultados


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

csv_file = Path("results/history_w2_e400.csv")
json_file = Path("results/summary_w2_e400.json")

df = pd.read_csv(csv_file)
with open(json_file, 'r', encoding='utf-8') as f:
    summary = json.load(f)

df.head()


## Resumen del experimento


In [ ]:
summary


In [ ]:
resumen_df = pd.DataFrame([{
    'Workers': summary['workers'],
    'Épocas': summary['epochs'],
    'Learning rate': summary['learning_rate'],
    'Arquitectura': str(summary['layer_dims']),
    'Train final': summary['final_train_acc'],
    'Test final': summary['final_test_acc'],
    'Tiempo total (s)': summary['total_time_sec'],
}])
resumen_df


## Historial completo


In [ ]:
df


## Gráfica: costo por época


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(df['epoch'], df['cost'], marker='o')
plt.title('Costo promedio por época')
plt.xlabel('Época')
plt.ylabel('Costo')
plt.grid(True)
plt.show()


## Gráfica: accuracy train vs test


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(df['epoch'], df['train'], marker='o', label='Train Accuracy')
plt.plot(df['epoch'], df['test'], marker='s', label='Test Accuracy')
plt.title('Accuracy de entrenamiento y prueba')
plt.xlabel('Época')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()


## Gráfica: tiempo por época


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(df['epoch'], df['epoch_time'], marker='o')
plt.title('Tiempo por época')
plt.xlabel('Época')
plt.ylabel('Segundos')
plt.grid(True)
plt.show()


## Gráfica: tiempo acumulado


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(df['epoch'], df['total_time'], marker='o')
plt.title('Tiempo acumulado')
plt.xlabel('Época')
plt.ylabel('Segundos')
plt.grid(True)
plt.show()


## Métricas derivadas


In [ ]:
gap = summary['final_train_acc'] - summary['final_test_acc']
avg_epoch_time = df['epoch_time'].mean()
best_test = df['test'].max()

metricas = pd.DataFrame([{
    'Gap train-test': gap,
    'Tiempo promedio por registro (s)': avg_epoch_time,
    'Mejor test accuracy registrado': best_test,
}])
metricas


## Conclusión automática


In [ ]:
print(f"Accuracy final en train: {summary['final_train_acc']:.4f}")
print(f"Accuracy final en test : {summary['final_test_acc']:.4f}")
print(f"Tiempo total           : {summary['total_time_sec']:.2f} s")
print(f"Gap train-test         : {summary['final_train_acc'] - summary['final_test_acc']:.4f}")

if (summary['final_train_acc'] - summary['final_test_acc']) < 0.02:
    print('Conclusión: el modelo generaliza bien y el entrenamiento fue estable.')
else:
    print('Conclusión: revisar posible sobreajuste.')
